# ⚽ Player Similarity Notebook

**Version:** MVP v0.1.0

Find statistically similar football players using FBref player statistics.

---

## Objective

This notebook allows you to:

- Load a player dataset
- Clean and validate the data
- Find statistically similar players
- Compare players using radar charts

---

## Workflow

1. Configuration
2. Load Dataset
3. Data Validation
4. Data Cleaning
5. Feature Selection
6. Player Similarity
7. Radar Chart
8. Export Results

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

## 1. Configuration

In [2]:
from src.config import DATA_RAW
from src.data_loader import load_dataset
from src.validation import validate_dataset

## 2. Load Dataset

In [3]:
dataset_path = DATA_RAW / "players_data_light-2024_2025.csv"

df = load_dataset(dataset_path)

validate_dataset(df)

df.head()

✓ Dataset validation successful.
Players: 2854
Required columns: OK
Similarity features: 7
Ready for similarity analysis.


,Rk,Player,Nation,Pos,Squad,Comp,Age,Born,MP,Starts,...,Att (GK),Thr,Launch%,AvgLen,Opp,Stp,Stp%,#OPA,#OPA/90,AvgDist
0,1,Max Aarons,eng ENG,DF,Bournemouth,eng Premier League,24.0,2000.0,3,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2,Max Aarons,eng ENG,"DF,MF",Valencia,es La Liga,24.0,2000.0,4,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3,Rodrigo Abajas,es ESP,DF,Valencia,es La Liga,21.0,2003.0,1,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,4,James Abankwah,ie IRL,"DF,MF",Udinese,it Serie A,20.0,2004.0,6,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,5,Keyliane Abdallah,fr FRA,FW,Marseille,fr Ligue 1,18.0,2006.0,1,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:
df[["Player", "Min", "90s", "Gls", "Ast", "xG", "xAG", "PrgC", "PrgP", "PrgR"]].head(10)

,Player,Min,90s,Gls,Ast,xG,xAG,PrgC,PrgP,PrgR
0,Max Aarons,86,1.0,0,0,0.0,0.0,1,8,3
1,Max Aarons,120,1.3,0,0,0.0,0.0,0,6,10
2,Rodrigo Abajas,65,0.7,0,0,0.1,0.0,3,2,3
3,James Abankwah,88,1.0,0,0,0.1,0.0,3,4,1
4,Keyliane Abdallah,3,0.0,0,0,0.0,0.0,1,0,0
5,Yunis Abdelhamid,1033,11.5,0,0,0.2,0.1,4,22,3
6,Himad Abdelli,2842,31.6,6,1,6.4,3.2,107,207,111
7,Mohamed Abdelmoneim,855,9.5,0,0,0.0,0.0,6,52,5
8,Ali Abdi,1393,15.5,5,2,4.3,1.9,35,42,101
9,Saud Abdulhamid,205,2.3,0,1,0.0,0.2,6,9,26


In [5]:
df[["Gls", "Ast", "xG", "xAG", "PrgC", "PrgP", "PrgR"]].describe()

,Gls,Ast,xG,xAG,PrgC,PrgP,PrgR
count,2854.000000,2854.000000,2854.000000,2854.000000,2854.000000,2854.000000,2854.000000
mean,1.682901,1.200771,1.706903,1.215662,20.733006,45.221093,44.796776
std,3.152732,1.946170,2.817612,1.686875,26.635816,50.147121,59.818947
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.000000,0.000000,0.100000,0.100000,2.000000,5.000000,3.000000
50%,0.000000,0.000000,0.700000,0.600000,11.000000,28.500000,20.000000
75%,2.000000,2.000000,2.100000,1.600000,29.000000,69.000000,66.000000
max,31.000000,18.000000,27.100000,14.200000,213.000000,362.000000,488.000000


In [8]:
from src.preprocessing import prepare_similarity_data

df_prepared = prepare_similarity_data(df)

print(f"Original players: {len(df)}")
print(f"Prepared players: {len(df_prepared)}")

Original players: 2854
Prepared players: 1570


In [9]:
per90_columns = [
    "Gls_per90",
    "Ast_per90",
    "xG_per90",
    "xAG_per90",
    "PrgC_per90",
    "PrgP_per90",
    "PrgR_per90",
]

df_prepared[
    ["Player", "Squad", "Min", "90s"] + per90_columns
].head()

,Player,Squad,Min,90s,Gls_per90,Ast_per90,xG_per90,xAG_per90,PrgC_per90,PrgP_per90,PrgR_per90
0,Yunis Abdelhamid,Saint-Étienne,1033,11.5,0.000000,0.000000,0.017391,0.008696,0.347826,1.913043,0.260870
1,Himad Abdelli,Angers,2842,31.6,0.189873,0.031646,0.202532,0.101266,3.386076,6.550633,3.512658
2,Ali Abdi,Nice,1393,15.5,0.322581,0.129032,0.277419,0.122581,2.258065,2.709677,6.516129
3,Abel,Osasuna,2074,23.0,0.086957,0.000000,0.021739,0.043478,2.173913,3.347826,4.000000
4,Matthis Abline,Nantes,2768,30.8,0.292208,0.064935,0.275974,0.123377,2.435065,1.558442,5.487013


In [10]:
df_prepared[per90_columns].isna().sum()

Gls_per90     0
Ast_per90     0
xG_per90      0
xAG_per90     0
PrgC_per90    0
PrgP_per90    0
PrgR_per90    0
dtype: int64